#Notebook da Camada Silver


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# criação dos nomes/caminhos dos schemas de cada camada
catalog = "vcredit_medallion"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"

#tabelas = ["chamados_hora", "base_atendentes", "base_motivos", "canais", "chamados", "clientes", "custos", "pesquisa_satisfacao"]


In [0]:
def normalizar_texto(col):
    r"""
    Normaliza texto:
    - tudo minúsculo
    - mantém só letras (com acento) e números
    - todo o resto (pontuação, símbolos, '_', \n, \t, \r e outros espaços) vira espaço
    - sequências de espaço viram um só, e trim nas pontas
    Aceita nome de coluna (str) ou Column.
    """
    c = F.col(col) if isinstance(col, str) else col
    return F.trim(
        F.regexp_replace(F.lower(c.cast("string")), r"[^\p{L}\p{M}0-9]+", " ")
    )

In [0]:
def normalizar_inteiro(col):
    """
    Remove tudo que não for dígito (0-9) e converte para inteiro (bigint).
    - texto sem nenhum dígito (ex.: 'NULL', '', 'abc') vira null
    - mais de 18 dígitos vira null (evita estouro do bigint)
    Aceita nome de coluna (str) ou Column.
    """
    c = F.col(col) if isinstance(col, str) else col
    digitos = F.regexp_replace(c.cast("string"), r"[^0-9]", "")
    return (
        F.when((digitos == "") | (F.length(digitos) > 18), F.lit(None).cast("long"))
         .otherwise(digitos.cast("long"))
    )

In [0]:
# dd/mm/aaaa  <qualquer coisa que não seja dígito>  hh:mm
# o \D+ cobre " às ", " �s ", " " etc. — não depende do encoding do "à"
PADRAO_DATAHORA = r"(\d{1,2})/(\d{1,2})/(\d{4})\D+(\d{1,2}):(\d{2})"

def extrair_datahora(col, prefixo):
    """
    Extrai dia, mês, ano, hora e minuto de strings tipo '12/07/2025 às 09:23:50'.
    Retorna um dict {f'{prefixo}_dia': Column, ...} para usar com df.withColumns().
    Se a string não bater com o padrão, as 5 partes vêm null.
    """
    c = F.col(col) if isinstance(col, str) else col
    s = c.cast("string")
    partes = ["dia", "mes", "ano", "hora", "minuto"]
    return {
        f"{prefixo}_{nome}": normalizar_inteiro(F.regexp_extract(s, PADRAO_DATAHORA, i))
        for i, nome in enumerate(partes, start=1)
    }